# Exercise 1A Transient heat diffusion toward a curved geothermal profile


## 1. problem setting

This notebook solves transient one-dimensional heat diffusion in a crustal column with radiogenic heat production. The initial temperature is evolved toward a curved steady geothermal profile.

**Governing equation**

$$\frac{\partial T}{\partial t}=\alpha\frac{\partial^2T}{\partial z^2}+S(z),\qquad S(z)=\frac{Q_0e^{-z/h_r}}{\rho c_p},\qquad \alpha=\frac{k}{\rho c_p}.$$

**Boundary and initial conditions**

$$T(0,t)=T_{top},\qquad T(L,t)=T_{bottom},\qquad T(z,0)=T_0(z).$$

**Parameter table**

| Symbol | Meaning | Value |
|---|---:|---:|
| $L$ | domain depth | 30 km |
| $n_z$ | grid nodes | 61 |
| $k$ | thermal conductivity | 2.8 W m$^{-1}$ K$^{-1}$ |
| $\rho$ | density | 2700 kg m$^{-3}$ |
| $c_p$ | heat capacity | 900 J kg$^{-1}$ K$^{-1}$ |
| $T_{top}$ | upper Dirichlet temperature | 293 K |
| $T_{bottom}$ | lower Dirichlet temperature | 873 K |
| $Q_0$ | surface radiogenic heat production | $1.0\times10^{-6}$ W m$^{-3}$ |
| $h_r$ | radiogenic decay depth | 10 km |
| CFL | explicit stability factor | 0.45 |
| $t_{end}$ | final time | 15 Myr |


In [ ]:
from pathlib import Path
from dataclasses import dataclass
from time import perf_counter
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.sparse import diags, eye, csr_matrix
from scipy.sparse.linalg import spsolve
from IPython.display import HTML, display

Plot_COLORS = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#F0E442", "#000000", "#7F7F7F", "#8B4513"]
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "axes.linewidth": 0.7,
    "axes.prop_cycle": plt.cycler(color=Plot_COLORS),
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "legend.fontsize": 6,
    "legend.frameon": False,
    "lines.linewidth": 1.1,
    "lines.markersize": 3,
    "image.cmap": "viridis",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "animation.embed_limit": 100,
})

SECONDS_PER_MYR = 365.25 * 24 * 3600 * 1e6

CASE_ID = "Exercise1A"
ROOT = Path.cwd()
FIG = ROOT / "figures" / CASE_ID
OUT = ROOT / "outputs" / CASE_ID
FIG.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

@dataclass
class Config:
                                                             
    L: float = 30e3
    nz: int = 61
    k: float = 2.8
    rho: float = 2700.0
    cp: float = 900.0
    T_top: float = 293.0                                       
    T_bottom: float = 873.0                                
    Q0: float = 1.0e-6                                                             
    hr: float = 10e3                                                                                
    cfl: float = 0.45                                                                  
    nsave: int = 50
    t_end_myr: float = 15.0

    @property
    def alpha(self):
        return self.k / (self.rho * self.cp)

cfg = Config()
z = np.linspace(0, cfg.L, cfg.nz)
z_km = z / 1000.0
dz = z[1] - z[0]
Q = cfg.Q0 * np.exp(-z / cfg.hr)
S = Q / (cfg.rho * cfg.cp)                                          
                                                                                            
S[0] = 0.0
S[-1] = 0.0
T0 = np.full_like(z, cfg.T_top, dtype=float); T0[-1] = cfg.T_bottom

def steady_exponential(z):
    A = cfg.Q0 * cfg.hr**2 / cfg.k
    C1 = (cfg.T_bottom - cfg.T_top - A * (1 - np.exp(-cfg.L / cfg.hr))) / cfg.L
    return cfg.T_top + C1 * z + A * (1 - np.exp(-z / cfg.hr))

Tsteady = steady_exponential(z)
TEMP_XLIM = (0.8 * min(np.min(T0), np.min(Tsteady)), 1.1 * max(np.max(T0), np.max(Tsteady)))

## 2. shared functions


In [ ]:
def Plot_axes(ax, grid=True):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", length=3, width=0.6, pad=2)
    if grid:
        ax.grid(True, color="0.88", linewidth=0.45, alpha=0.8)
    return ax

def linf(u, ref):
    return float(np.max(np.abs(np.asarray(u, dtype=float) - np.asarray(ref, dtype=float))))

def choose_snapshot_steps(nsteps, nsave):
    return set(np.unique(np.round(np.linspace(0, nsteps, min(nsave, nsteps + 1))).astype(int)))

def plot_snapshots_depth_temperature(z_km, snapshots, times, steady, fname, title,
                                     time_scale=1.0, time_label="", xlim=None):
    snapshots = np.asarray(snapshots)
    times = np.asarray(times)
    idx = np.unique(np.round(np.linspace(0, len(times) - 1, min(10, len(times)))).astype(int))
    fig, axes = plt.subplots(2, 5, figsize=(7.2, 3.5), sharex=True, sharey=True)
    panel_labels = list("abcdefghij")
    for k, ax in enumerate(axes.flat):
        if k < len(idx):
            j = idx[k]
            ax.plot(snapshots[j], z_km, color=Plot_COLORS[0], lw=1.15)
            ax.plot(steady, z_km, color="0.15", ls="--", lw=0.9)
            ax.invert_yaxis()
            if xlim is not None:
                ax.set_xlim(*xlim)
            ax.set_ylim(z_km[-1], z_km[0])
            ax.set_title(f"t={times[j]/time_scale:.3g}{time_label}", pad=2)
            Plot_axes(ax)
            ax.text(0.03, 0.94, f"({panel_labels[k]})", transform=ax.transAxes,
                    ha="left", va="top", fontsize=7, fontweight="bold")
        else:
            ax.axis("off")
    for ax in axes[-1, :]: ax.set_xlabel("T [K]")
    for ax in axes[:, 0]: ax.set_ylabel("z [km]")
    fig.suptitle(title, y=1.02, fontsize=8)
    fig.tight_layout(w_pad=0.8, h_pad=0.9)
    fig.savefig(fname)
    plt.close(fig)
    return Path(fname)

def animate_depth_temperature(z_km, snapshots, times, steady, title,
                              time_scale=1.0, time_label="", xlim=None, save_path=None):
    """Create a template-style temperature-depth animation and optionally save it as a GIF."""
    snapshots = np.asarray(snapshots)
    times = np.asarray(times)
    fig, ax = plt.subplots(figsize=(3.2, 3.6))
    ax.plot(steady, z_km, "k--", lw=1.2, label="steady state")
    line, = ax.plot(snapshots[0], z_km, lw=2.0, label="numerical")
    ax.invert_yaxis()
    if xlim is not None:
        ax.set_xlim(*xlim)
    ax.set_ylim(z_km[-1], z_km[0])
    ax.set_xlabel("Temperature T [K]")
    ax.set_ylabel("Depth z [km]")
    ax.set_title(title)
    Plot_axes(ax)
    ax.legend(fontsize=8)
    time_text = ax.text(0.03, 0.03, "", transform=ax.transAxes)
    def update(i):
        line.set_data(snapshots[i], z_km)
        ax.set_title(f"{title}; t={times[i]/time_scale:.3g}{time_label}")
        time_text.set_text(f"t = {times[i]/time_scale:.3g}{time_label}")
        return line, time_text
    anim = FuncAnimation(fig, update, frames=len(times), interval=120, blit=True)
    if save_path is not None:
        anim.save(save_path, writer=PillowWriter(fps=5), dpi=90)
    plt.close(fig)
    return anim

def display_animation(animation):
    """Display an animation in the notebook with playback controls."""
    try:
        display(HTML(animation.to_jshtml()))
    except OSError as exc:
        print(f"Animation display skipped: {exc}")

def _png_data_uri(path):
    """Embed a saved PNG in HTML so the row display works even with absolute paths."""
    path = Path(path)
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:image/png;base64,{encoded}"

def display_method_result_row(method_label, snapshot_path, animation):
    """
    The animation is inserted via animation.to_jshtml(), so the notebook keeps
    the play/pause buttons and frame slider
    """
    snapshot_uri = _png_data_uri(snapshot_path)
    anim_html = animation.to_jshtml()

    row_html = f"""
    <div style="margin: 16px 0 28px 0; width: 100%;">
      <div style="font-weight: 700; font-size: 16px; margin-bottom: 8px;">{method_label}</div>
      <div style="display: flex; flex-direction: row; gap: 18px; align-items: flex-start; width: 100%;">
        <div style="flex: 1 1 0; min-width: 0;">
          <div style="font-weight: 600; margin-bottom: 4px;">Snapshots</div>
          <img src="{snapshot_uri}" style="width: 100%; height: auto; display: block;">
        </div>
        <div style="flex: 1 1 0; min-width: 0; overflow-x: auto;">
          <div style="font-weight: 600; margin-bottom: 4px;">Animation controls</div>
          {anim_html}
        </div>
      </div>
    </div>
    """
    display(HTML(row_html))

def plot_residual_history(times, errors, fname, title, ylabel="max |T - Tsteady| [K]", time_scale=1.0, time_label=""):
    fig, ax = plt.subplots(figsize=(6, 4))
    for name, vals in errors.items():
        ax.semilogy(np.asarray(times[name]) / time_scale, np.maximum(vals, 1e-14), label=name)
    ax.set_xlabel(f"time{time_label}")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    Plot_axes(ax)
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(fname)
    plt.show()

def impose_dirichlet(T):
    T = np.asarray(T, dtype=float).copy()
    T[0] = cfg.T_top
    T[-1] = cfg.T_bottom
    return T

def steady_residual_history(snapshots):
    """Residual with respect to the steady profile, measured in Kelvin."""
    return np.array([linf(s, Tsteady) for s in snapshots])

def heat_residual_summary(method, snapshots, times, elapsed, dt, nsteps):
    max_res = steady_residual_history(snapshots)
    return {
        "method": method,
        "dt_MYR": dt / SECONDS_PER_MYR,                                                        
        "dt_Myr": dt / SECONDS_PER_MYR,
        "steps": int(nsteps),
        "wall_time_s": elapsed,
        "time_per_step_s": elapsed / max(nsteps, 1),
        "final_max_steady_residual_K": float(max_res[-1]),
        "max_steady_residual_history_K": max_res,
    }

def save_heat_method(method_key, method_label, snapshots, times, elapsed, dt, nsteps, store):
    method_xlim = TEMP_XLIM
    store[method_key] = heat_residual_summary(method_label, snapshots, times, elapsed, dt, nsteps)
    store[method_key]["snapshots"] = snapshots
    store[method_key]["times"] = times
    snapshot_path = plot_snapshots_depth_temperature(
        z_km, snapshots, times, Tsteady,
        FIG / f"{method_key}_snapshots.png",
        f"{method_label}: snapshots", time_scale=SECONDS_PER_MYR, time_label=" Myr",
        xlim=method_xlim,
    )
    anim = animate_depth_temperature(
        z_km, snapshots, times, Tsteady,
        f"{method_label}: relaxation animation", time_scale=SECONDS_PER_MYR, time_label=" Myr",
        xlim=method_xlim, save_path=FIG / f"{method_key}_animation.gif",
    )
    display_method_result_row(method_label, snapshot_path, anim)

fig, ax = plt.subplots(figsize=(3.2, 3.6))
ax.plot(Tsteady, z_km, label="steady geotherm")
ax.plot(T0, z_km, "--", label="cold initial geotherm")
ax.invert_yaxis()
ax.set_xlim(*TEMP_XLIM)
ax.set_ylim(z_km[-1], z_km[0])
ax.set_xlabel("Temperature T [K]")
ax.set_ylabel("Depth z [km]")
ax.set_title("Exercise 1A setup")
Plot_axes(ax)
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "setup_steady_and_initial.png")
plt.show()
results = {}

## 3. FDM explicit


In [ ]:
dt_explicit_fdm_eval = cfg.cfl * dz**2 / cfg.alpha
t_end = cfg.t_end_myr * SECONDS_PER_MYR
nsteps_explicit_fdm_eval = int(np.ceil(t_end / dt_explicit_fdm_eval))
dt_explicit_fdm_eval = t_end / nsteps_explicit_fdm_eval
                                                                                                                    
def fem_mk(n, h):
    # Assemble linear-element mass and stiffness matrices.
    # Keep boundary coupling terms for the interior solve.
    M=np.zeros((n,n)); K=np.zeros((n,n))
    Me=h/6*np.array([[2,1],[1,2]], float); Ke=1/h*np.array([[1,-1],[-1,1]], float)
    for e in range(n-1):
        sl=slice(e,e+2); M[sl,sl]+=Me; K[sl,sl]+=Ke
    return M, K

def fem_explicit_stable_dt(Mii, Kii, safety=0.45):
    # Estimate the largest stable explicit FEM step from the operator spectrum.
    """Explicit FEM timestep from the largest eigenvalue of M^{-1}K."""
    eigvals = np.linalg.eigvals(np.linalg.solve(Mii, Kii))
    lambda_max = np.max(np.real(eigvals))
    return min(safety * 2.0 / (cfg.alpha * lambda_max), t_end)

M_fem_base, K_fem_base = fem_mk(cfg.nz, dz)
interior = slice(1, -1)
Kii_fem = K_fem_base[interior, interior]
Kbc_fem = K_fem_base[1:-1, 0] * cfg.T_top + K_fem_base[1:-1, -1] * cfg.T_bottom

Mii_fem_consistent = M_fem_base[interior, interior]
M_fem_lumped_base = np.diag(M_fem_base.sum(axis=1))
Mii_fem_lumped = M_fem_lumped_base[interior, interior]
F_fem_consistent = (M_fem_base @ S)[interior]
F_fem_lumped = (M_fem_lumped_base @ S)[interior]

dt_explicit_fem_consistent_eval = fem_explicit_stable_dt(Mii_fem_consistent, Kii_fem)

dt_explicit_fem_lumped_eval = fem_explicit_stable_dt(Mii_fem_lumped, Kii_fem)
                                                                                                                                             
dt_common = min(dt_explicit_fdm_eval, dt_explicit_fem_consistent_eval, dt_explicit_fem_lumped_eval)
nsteps_common = int(np.ceil(t_end / dt_common))
dt_common = t_end / nsteps_common
dt_explicit_fdm = dt_common
nsteps_explicit_fdm = nsteps_common
dt_explicit_fem_consistent = dt_common                                                                                    
nsteps_explicit_fem_consistent = nsteps_common                                                                 
dt_explicit_fem_lumped = dt_common                                                                                    
nsteps_explicit_fem_lumped = nsteps_common                                                                 
print(f"Common minimum dt = {dt_common:.6e} s ({dt_common / SECONDS_PER_MYR:.6e} Myr), steps = {nsteps_common}")
print(f"FEM explicit dt eval, consistent mass = {dt_explicit_fem_consistent_eval:.6e} s; using common dt = {dt_explicit_fem_consistent:.6e} s, steps = {nsteps_explicit_fem_consistent}")                                                      
print(f"FEM explicit dt eval, lumped mass     = {dt_explicit_fem_lumped_eval:.6e} s; using common dt = {dt_explicit_fem_lumped:.6e} s, steps = {nsteps_explicit_fem_lumped}")                                                      

In [ ]:
def fdm_explicit():
    # Build the finite-difference Laplacian on interior unknowns.
    # Advance with the explicit update using the CFL-limited time step.
    T = impose_dirichlet(T0); save_steps = choose_snapshot_steps(nsteps_explicit_fdm, cfg.nsave)
    snapshots = []; times = []
    n = cfg.nz-2; D = diags([np.ones(n-1), -2*np.ones(n), np.ones(n-1)], [-1,0,1], format="csr")/dz**2
    bc=np.zeros(n); bc[0]+=cfg.alpha*cfg.T_top/dz**2; bc[-1]+=cfg.alpha*cfg.T_bottom/dz**2
    I = eye(n, format="csr")
    A = I + dt_explicit_fdm*cfg.alpha*D
    tic = perf_counter()

    for step in range(nsteps_explicit_fdm+1):
        if step in save_steps: snapshots.append(T.copy()); times.append(step*dt_explicit_fdm)
        if step == nsteps_explicit_fdm: break
        rhs = A*T[1:-1] + dt_explicit_fdm*(S[1:-1] + bc)
        T[1:-1]=rhs; T=impose_dirichlet(T)
    return np.array(snapshots), np.array(times), perf_counter()-tic, dt_explicit_fdm, nsteps_explicit_fdm

snap, tt, elapsed, dt_explicit_fdm, ns = fdm_explicit()
print(f"FDM explicit Euler elapsed time: {elapsed:.6f} s")
save_heat_method("fdm_explicit", "FDM explicit Euler", snap, tt, elapsed, dt_explicit_fdm, ns, results)


## 4. FDM implicit


In [ ]:
def fdm_implicit():
    # Assemble the backward-Euler system matrix for one implicit time step.
    # Solve the linear system at each time level before applying boundary values.
    dt = dt_common; nsteps = nsteps_common                                                                      
    n=cfg.nz-2; D=diags([np.ones(n-1), -2*np.ones(n), np.ones(n-1)], [-1,0,1], format="csr")/dz**2
    I = eye(n, format="csr")
    A = I - dt*cfg.alpha*D
    bc=np.zeros(n); bc[0]+=cfg.alpha*cfg.T_top/dz**2; bc[-1]+=cfg.alpha*cfg.T_bottom/dz**2
    T=impose_dirichlet(T0); save_steps=choose_snapshot_steps(nsteps, cfg.nsave); snapshots=[]; times=[]
    tic=perf_counter()
    for step in range(nsteps+1):
        if step in save_steps: snapshots.append(T.copy()); times.append(step*dt)
        if step == nsteps: break
        rhs = T[1:-1] + dt*(S[1:-1] + bc)
        T[1:-1]=spsolve(A, rhs); T=impose_dirichlet(T)
    return np.array(snapshots), np.array(times), perf_counter()-tic, dt, nsteps
snap, tt, elapsed, dt, ns = fdm_implicit()
print(f"FDM backward Euler elapsed time: {elapsed:.6f} s")
save_heat_method("fdm_backward_euler", "FDM backward Euler", snap, tt, elapsed, dt, ns, results)


## 5. FDM Crank--Nicolson


In [ ]:
def fdm_crank_nicolson():
    # Assemble Crank--Nicolson left and right time-stepping matrices.
    # Use midpoint diffusion/wave weighting for second-order time accuracy.
    dt = dt_common; nsteps = nsteps_common                                                                      
    n=cfg.nz-2; D=diags([np.ones(n-1), -2*np.ones(n), np.ones(n-1)], [-1,0,1], format="csr")/dz**2
    I = eye(n, format="csr")
    A = I - 0.5*dt*cfg.alpha*D
    B = I + 0.5*dt*cfg.alpha*D
    bc=np.zeros(n); bc[0]+=cfg.alpha*cfg.T_top/dz**2; bc[-1]+=cfg.alpha*cfg.T_bottom/dz**2
    T=impose_dirichlet(T0); save_steps=choose_snapshot_steps(nsteps, cfg.nsave); snapshots=[]; times=[]
    tic=perf_counter()
    for step in range(nsteps+1):
        if step in save_steps: snapshots.append(T.copy()); times.append(step*dt)
        if step == nsteps: break
        rhs = B @ T[1:-1] + dt*(S[1:-1] + bc)
        T[1:-1]=spsolve(A, rhs); T=impose_dirichlet(T)
    return np.array(snapshots), np.array(times), perf_counter()-tic, dt, nsteps
snap, tt, elapsed, dt, ns = fdm_crank_nicolson()
print(f"FDM Crank--Nicolson elapsed time: {elapsed:.6f} s")
save_heat_method("fdm_crank_nicolson", "FDM Crank--Nicolson", snap, tt, elapsed, dt, ns, results)

## 6. Method 4 — FEM explicit, consistent and lumped mass


In [ ]:
def fem_explicit_consistent_and_lumped_mass(label, lumped=False):
    # Select either the consistent or lumped FEM mass matrix.
    # Use the FEM stability estimate to set the explicit time step.
    if lumped:
        Mii = Mii_fem_lumped
        F = F_fem_lumped
        dt_explicit_fem = dt_explicit_fem_lumped
        nsteps_explicit_fem = nsteps_explicit_fem_lumped
        lump_diag = np.diag(Mii)
    else:
        Mii = Mii_fem_consistent
        F = F_fem_consistent
        dt_explicit_fem = dt_explicit_fem_consistent
        nsteps_explicit_fem = nsteps_explicit_fem_consistent
        lump_diag = None

    T = impose_dirichlet(T0); save_steps = choose_snapshot_steps(nsteps_explicit_fem, cfg.nsave)
    snapshots = []; times = []; tic = perf_counter()

    for step in range(nsteps_explicit_fem + 1):
        if step in save_steps: snapshots.append(T.copy()); times.append(step * dt_explicit_fem)
        if step == nsteps_explicit_fem: break
        u = T[1:-1]
        rhs = Mii@ u + dt_explicit_fem * (F - cfg.alpha * (Kii_fem @ u + Kbc_fem))
        T[1:-1] = np.linalg.solve(Mii, rhs)
        T = impose_dirichlet(T)
    return np.array(snapshots), np.array(times), perf_counter() - tic, dt_explicit_fem, nsteps_explicit_fem

for key, label, lumped in [
    ("fem_explicit_consistent", "FEM explicit Euler, consistent mass", False),
    ("fem_explicit_lumped", "FEM explicit Euler, lumped mass", True),
]:
    snap, tt, elapsed, dt, ns = fem_explicit_consistent_and_lumped_mass(label, lumped=lumped)
    print(f"{label} elapsed time: {elapsed:.6f} s")
    save_heat_method(key, label, snap, tt, elapsed, dt, ns, results)

## 7. Method 5 — FEM backward, consistent and lumped mass


In [ ]:
def fem_backward_consistent_and_lumped_mass(label, lumped=False):
    # Select either the consistent or lumped FEM mass matrix.
    # Assemble the backward FEM matrix or block system for implicit stepping.
    if lumped:
        Mii = Mii_fem_lumped
        F = F_fem_lumped
        dt_ref_fem = dt_explicit_fem_lumped
        lump_diag = np.diag(Mii)
    else:
        Mii = Mii_fem_consistent
        F = F_fem_consistent
        dt_ref_fem = dt_explicit_fem_consistent
        lump_diag = None

    dt_fem = dt_common                                                                                    
    nsteps_fem = nsteps_common
    A = Mii + dt_fem * cfg.alpha * Kii_fem
    T = impose_dirichlet(T0); save_steps = choose_snapshot_steps(nsteps_fem, cfg.nsave)
    snapshots = []; times = []; tic = perf_counter()
    for step in range(nsteps_fem + 1):
        if step in save_steps: snapshots.append(T.copy()); times.append(step * dt_fem)
        if step == nsteps_fem: break
        rhs = Mii @ T[1:-1] + dt_fem * (F - cfg.alpha * Kbc_fem)
        T[1:-1] = np.linalg.solve(A, rhs)
        T = impose_dirichlet(T)
    return np.array(snapshots), np.array(times), perf_counter() - tic, dt_fem, nsteps_fem

for key, label, lumped in [
    ("fem_implicit_consistent", "FEM backward Euler, consistent mass", False),
    ("fem_implicit_lumped", "FEM backward Euler, lumped mass", True),
]:
    snap, tt, elapsed, dt, ns = fem_backward_consistent_and_lumped_mass(label, lumped)
    print(f"{label} elapsed time: {elapsed:.6f} s")
    save_heat_method(key, label, snap, tt, elapsed, dt, ns, results)


## 8. Method 6 — pseudospectral sine method for heat or wave equation


In [ ]:
nsteps = nsteps_common                                                                            
dt = dt_common                                        

def pseudospectral_sine_method():
    # Transform the initial field and source into modal coefficients.
    # Advance spectral modes independently in time before reconstructing the field.
    theta0 = impose_dirichlet(T0) - Tsteady; theta0_int = theta0[1:-1]
    n_int = cfg.nz - 2; modes = np.arange(1, n_int + 1)
    i = np.arange(1, n_int + 1)[:, None]
    m = modes[None, :]
                               
    Phi = np.sin(np.pi * i * m / (n_int + 1))
    coeff = (2.0 / (n_int + 1)) * (Phi.T @ theta0_int)
    lam = (modes * np.pi / cfg.L) ** 2
    nsteps = nsteps_common; dt = dt_common                                                                            
    save_steps = choose_snapshot_steps(nsteps, cfg.nsave)                                                     
    amp = np.exp(-cfg.alpha * lam * dt)
    snapshots = []; times = []
    coeff_t = coeff.copy()
    t = 0.0                                                               
    theta = Phi @ coeff_t                                                       
    T = Tsteady.copy()                                                                      
    T[1:-1] += theta                                                      
    tic = perf_counter()
    for step in range(nsteps + 1):
        if step in save_steps:        
            snapshots.append(impose_dirichlet(T))
            times.append(t)
        if step == nsteps: break
        coeff_t = amp * coeff_t
        t = (step + 1) * dt                                                                   
        theta = Phi @ coeff_t
        T = Tsteady.copy()
        T[1:-1] += theta
    return np.array(snapshots), np.array(times), perf_counter() - tic, dt, nsteps

snap, tt, elapsed, dt, ns = pseudospectral_sine_method()
print(f"Pseudospectral sine elapsed time: {elapsed:.6f} s")
save_heat_method("pseudospectral_sine_method", "Pseudospectral sine method", snap, tt, elapsed, dt, ns, results)

## 9. Final residual


In [ ]:
summary = pd.DataFrame([
    {k: v for k, v in d.items() if not isinstance(v, np.ndarray) and k not in ["snapshots", "times"]}
    for d in results.values()
]).sort_values("final_max_steady_residual_K")

display(summary)
summary.to_csv(OUT / "exercise1a_residual_efficiency_summary.csv", index=False)
                     
residual_hist = {v["method"]: v["max_steady_residual_history_K"] for v in results.values()}
time_hist = {v["method"]: v["times"] for v in results.values()}
plot_residual_history(
    time_hist,
    residual_hist,
    FIG / "exercise1a_max_steady_residual_history.png",
    "Exercise 1A: convergence to steady state",
    "max |T - Tsteady| [K]",
    time_scale=SECONDS_PER_MYR,
    time_label=" [Myr]",
)
                                                                          
fig, ax = plt.subplots(figsize=(6, 4))
for key, v in results.items():
    ax.plot(v["snapshots"][-1] - Tsteady, z_km, label=v["method"])
ax.axvline(0.0, color="k", lw=0.7)
ax.invert_yaxis()
ax.set_ylim(z_km[-1], z_km[0])
ax.set_xlabel("T - Tsteady [K]")
ax.set_ylabel("Depth z [km]")
ax.set_title("Final steady residual profile")
Plot_axes(ax)
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "exercise1a_final_residual_profiles_combined.png")
plt.show()
                                                                                            
n_methods = len(results)
ncols = 3
nrows = int(np.ceil(n_methods / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 4.8 * nrows), sharey=True)
axes = np.atleast_1d(axes).ravel()
for ax, (key, v) in zip(axes, results.items()):
    residual = v["snapshots"][-1] - Tsteady
    ax.plot(residual, z_km)
    ax.axvline(0.0, color="k", lw=0.7)
    ax.invert_yaxis()
    ax.set_ylim(z_km[-1], z_km[0])
    ax.set_title(v["method"], fontsize=9)
    ax.set_xlabel("T - Tsteady [K]")
    Plot_axes(ax)
for ax in axes[::ncols]:
    ax.set_ylabel("Depth z [km]")
for ax in axes[n_methods:]:
    ax.axis("off")
fig.suptitle("Exercise 1A: final residual profile by method", y=1.01)
fig.tight_layout()
fig.savefig(FIG / "exercise1a_final_residual_profiles_by_method.png")
plt.show()

print("Summary CSV saved to:", OUT / "exercise1a_residual_efficiency_summary.csv")
print("Figures and GIF animations saved in:", FIG)

## 10. L2 error


In [ ]:
def green_DD_heat_reference(times, nmodes=None):
    """
    Explicit Green-function reference for Exercise 1A.

    PDE for perturbation:
        theta_t = alpha theta_zz
        theta(0,t) = theta(L,t) = 0

    Green basis:
        sin(n*pi*z/L), n = 1,2,...

    T(z,t) = Tsteady(z) + sum_n a_n sin(n*pi*z/L)
             exp[-alpha*(n*pi/L)^2 t]
    """
    times = np.asarray(times, dtype=float)

    if nmodes is None:
        nmodes = cfg.nz - 2

    theta0 = impose_dirichlet(T0) - Tsteady

    n = np.arange(1, nmodes + 1)
    k = n * np.pi / cfg.L

    Phi = np.sin(np.outer(z, k))
                                                                             
    coeff = (2.0 / cfg.L) * np.trapezoid(theta0[:, None] * Phi, x=z, axis=0)

    tic = perf_counter()
    snapshots = []

    for t in times:
        theta = Phi @ (coeff * np.exp(-cfg.alpha * k**2 * t))
        T = impose_dirichlet(Tsteady + theta)
        snapshots.append(T)

    elapsed = perf_counter() - tic
    return np.asarray(snapshots), elapsed

rows = []

for key, v in results.items():
    times = np.asarray(v["times"], dtype=float)
    T_num = np.asarray(v["snapshots"], dtype=float)

    T_ref, ref_elapsed = green_DD_heat_reference(times)
    diff = T_num - T_ref

    l2_t = np.sqrt(np.trapezoid(diff**2, x=z, axis=1))
    ref_l2_t = np.sqrt(np.trapezoid(T_ref**2, x=z, axis=1))
    rel_l2_t = l2_t / np.maximum(ref_l2_t, 1e-300)

    rows.append({
        "method": v["method"],
        "dt": v.get("dt_MYR", np.nan),                                                                             
        "steps": v.get("steps", np.nan),
        "wall time": v.get("wall_time_s", np.nan),
        "final rel L2": float(rel_l2_t[-1]),
        "max rel L2": float(np.max(rel_l2_t)),
    })


green_l2_summary = pd.DataFrame(rows)[["method", "dt", "steps", "wall time", "final rel L2", "max rel L2"]].sort_values(
    "max rel L2"
)

display(green_l2_summary)

green_l2_summary.to_csv(
    OUT / "exercise1a_l2_error_vs_green_DD.csv",
    index=False
)
